# 14 — Adaptive Query Execution (AQE)

**What you will learn:**
- What AQE is and why it was introduced (Spark 3.0+)
- Feature 1: Dynamically coalescing shuffle partitions
- Feature 2: Dynamically switching join strategies at runtime
- Feature 3: Dynamically optimizing skew joins
- Key AQE configuration parameters
- How to confirm AQE decisions in the query plan
- AQE in Databricks (always-on)

## 1. What is AQE?

**Before Spark 3.0:** Query plan was built **once upfront** using *estimated* statistics — often wrong.

**With AQE (Spark 3.0+):** Spark **re-optimizes the plan at runtime** after each shuffle stage using *actual* measured statistics.

```
Static plan:  optimize once upfront ──────────────────────► run
AQE plan:     optimize → run stage 1 → re-optimize → run stage 2 → re-optimize → ...
                                 ↑ uses real row counts, sizes, skew info
```

Three core optimizations AQE applies automatically:
1. **Coalesce shuffle partitions** — merge tiny post-shuffle partitions
2. **Switch join strategy** — promote Sort-Merge → Broadcast at runtime
3. **Skew join optimization** — split skewed partitions automatically

In [ ]:
# Check AQE status (default: true in Spark 3.0+, always-on in Databricks)
print("AQE enabled:             ", spark.conf.get("spark.sql.adaptive.enabled"))
print("Coalesce partitions:     ", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
print("Skew join:               ", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

## 2. Feature 1 — Dynamically Coalescing Shuffle Partitions

**Problem:** `shuffle.partitions = 200` (default) produces 200 output partitions — even for a 1 MB result. 200 tiny tasks, huge scheduler overhead.

**AQE Fix:** After each shuffle stage completes, AQE measures actual partition sizes and **merges adjacent small partitions** into larger ones.

You no longer need to hand-tune `shuffle.partitions` — set it high and AQE will coalesce down.

In [ ]:
import pyspark.sql.functions as F

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Target size for coalesced partitions (default 64 MB)
print("Target partition size (bytes):",
      spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes", "67108864"))

# Set shuffle.partitions high — AQE coalesces down automatically
spark.conf.set("spark.sql.shuffle.partitions", "200")

df = spark.createDataFrame(
    [(i, ["Eng","Mkt","HR"][i%3], float(i*100)) for i in range(1, 10001)],
    ["id","dept","salary"]
)

result = df.groupBy("dept").agg(F.sum("salary"))
result.explain(mode="formatted")  # look for AQEShuffleRead with coalesced partitions
result.show()

## 3. Feature 2 — Dynamic Join Strategy Switching

**Problem:** At plan time, Spark does not know the true size of a filtered table and conservatively picks Sort-Merge Join.

**AQE Fix:** After the build-side stage completes, AQE checks the **actual output size**. If it fits in the broadcast threshold, AQE **switches to Broadcast Hash Join** automatically — even without a `broadcast()` hint.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

large_orders = spark.createDataFrame(
    [(i, i % 5000, float(i*10)) for i in range(1, 500001)],
    ["order_id","customer_id","amount"]
)

# Only 100 rows — after filtering, AQE detects this is small enough to broadcast
small_customers = spark.createDataFrame(
    [(i, f"VIP_{i}") for i in range(1, 101)],
    ["customer_id","name"]
)

joined = large_orders.join(small_customers, on="customer_id")
joined.explain(mode="formatted")  # AQE should show BroadcastHashJoin without hint
print("Count:", joined.count())

## 4. Feature 3 — Dynamic Skew Join Optimization

**Problem:** After shuffling, one partition has 10× more data than others → one task is the straggler — the whole stage waits.

**AQE Fix:** After the shuffle stage, AQE **detects oversized partitions** and splits them into smaller sub-tasks. Each sub-task is joined with a copy of the matching partition from the other side.

In [ ]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("Skew factor (vs median):",
      spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5"))
print("Skew min threshold (bytes):",
      spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "268435456"))

# Simulate: key=1 is 90% of the left side
skewed_left = spark.createDataFrame(
    [(1, f"order_{i}") for i in range(1, 90001)] +
    [(i % 100 + 2, f"order_{i}") for i in range(90001, 100001)],
    ["key","order_name"]
)
right = spark.createDataFrame(
    [(i, f"item_{i}") for i in range(1, 201)], ["key","item_name"]
)

result = skewed_left.join(right, on="key")
result.explain(mode="formatted")   # look for CustomShuffleReader handling the skew
print("Count:", result.count())

## 5. AQE Configuration Reference

| Parameter | Default | Description |
|---|---|---|
| `spark.sql.adaptive.enabled` | `true` | Master switch for AQE |
| `spark.sql.adaptive.coalescePartitions.enabled` | `true` | Merge small shuffle partitions |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | 64 MB | Target size after coalescing |
| `spark.sql.adaptive.coalescePartitions.minPartitionNum` | 1 | Minimum partitions after merge |
| `spark.sql.adaptive.skewJoin.enabled` | `true` | Auto-split skewed partitions |
| `spark.sql.adaptive.skewJoin.skewedPartitionFactor` | 5 | N× the median size = skewed |
| `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` | 256 MB | Min absolute size to consider skewed |

In [ ]:
# Recommended AQE config for production
spark.conf.set("spark.sql.adaptive.enabled",                          "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",       "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes",     str(128 * 1024 * 1024))
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",                 "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor",   "5")
print("AQE production config applied.")

## 6. AQE in Databricks

- AQE is **enabled by default** on Databricks Runtime 7.3+
- Databricks adds **Photon** (native vectorized engine) on top of AQE
- **Enhanced AQE** in Databricks extends standard AQE with:
  - Better statistics from Delta Lake file-level min/max
  - Auto-tuned skew thresholds per cluster
  - Integration with Photon for vectorized shuffle reads
- You generally **do not need to tune AQE** in Databricks — defaults are well-calibrated

## Key Takeaways

| AQE Feature | Problem Solved | Mechanism |
|---|---|---|
| Coalesce shuffle partitions | 200 tiny partitions from shuffle | Merges adjacent small partitions post-shuffle |
| Dynamic join switch | Static plan chose Sort-Merge; actual data is small | Switches to BHJ after measuring real size |
| Skew join optimization | One partition 10× larger = straggler | Splits skewed partition into sub-tasks |

**In Databricks:** AQE is on by default — trust it. Tune only when Spark UI shows specific bottlenecks.